In [19]:
import pandas as pd

# Cargamos el CSV (asegúrate de que el nombre coincida con tu archivo)
df = pd.read_csv('stock_atb.csv')


# Creamos un diccionario con los cambios
# Es buena práctica usar minúsculas y guiones bajos (snake_case) para programar rápido
nuevos_nombres = {
    'Marca temporal': 'timestamp',
    'Tipo de registro': 'tipo_registro',
    'Antibiótico': 'antibiotico',
    'Responsable (iniciales)': 'responsable',
    'Fecha de recepción': 'fecha_recepcion',
    'Cantidad (En caso de sensidiscos, indicar cantidad de tubos)': 'cantidad',
    'Fecha de apertura': 'fecha_apertura',
    'Fecha de cierre': 'fecha_cierre',
    'Motivo de cierre': 'motivo_cierre',
    'Comentario (opcional)': 'comentario',
    'Fecha de vencimiento': 'fecha_vencimiento',
    'Cantidad eliminada': 'cantidad_eliminada'
}

# Aplicamos el cambio
df = df.rename(columns=nuevos_nombres)

# 1. Rellenar los comentarios vacíos para que no se vea el NaN
df['comentario'] = df['comentario'].fillna('Sin obs.')

# 2. Rellenar la cantidad eliminada con 0 (porque si es NaN, es que no se eliminó nada)
df['cantidad_eliminada'] = df['cantidad_eliminada'].fillna(0)

# 3. Verifiquemos cómo quedó la columna 'cantidad_eliminada'
# print(df['comentario'].head())

# Ver solo las primeras 5 filas de las columnas clave
# print(df.isnull().sum())

# Lista de columnas que queremos fuera
columnas_a_eliminar = ['timestamp', 'responsable', 'comentario']

# Eliminamos con precaución
df = df.drop(columns=columnas_a_eliminar, errors='ignore')

# Verificamos qué quedó realmente
print("Columnas actuales en el DataFrame:")
print(df.columns.tolist())

Columnas actuales en el DataFrame:
['tipo_registro', 'antibiotico', 'fecha_recepcion', 'cantidad', 'fecha_apertura', 'fecha_cierre', 'motivo_cierre', 'fecha_vencimiento', 'cantidad_eliminada']


In [28]:
import pandas as pd

# 1. CARGA DE DATOS
file_name = 'stock_atb.csv'
df = pd.read_csv(file_name)

# 2. RENOMBRAR COLUMNAS
nuevos_nombres = {
    'Tipo de registro': 'tipo_registro',
    'Antibiótico': 'antibiotico',
    'Fecha de recepción': 'fecha_recepcion',
    'Cantidad (En caso de sensidiscos, indicar cantidad de tubos)': 'cantidad',
    'Fecha de apertura': 'fecha_apertura',
    'Fecha de cierre': 'fecha_cierre',
    'Motivo de cierre': 'motivo_cierre',
    'Fecha de vencimiento': 'fecha_vencimiento',
    'Cantidad eliminada': 'cantidad_eliminada'
}
df = df.rename(columns=nuevos_nombres)

# 3. ELIMINACIÓN DE COLUMNAS NO RELEVANTES (Timestamp, Responsable, Comentario)
# Usamos errores='ignore' por si ya las habías borrado antes
columnas_no_relevantes = ['Marca temporal', 'Responsable (iniciales)', 'Comentario (opcional)']
df = df.drop(columns=columnas_no_relevantes, errors='ignore')

# 4. TRATAMIENTO DE FECHAS
# 'dayfirst=True' es vital para el formato chileno (DD/MM/AAAA)
# 'errors=coerce' transforma lo que no es fecha en NaT
cols_fechas = ['fecha_recepcion', 'fecha_apertura', 'fecha_cierre', 'fecha_vencimiento']
for col in cols_fechas:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')

# 5. LIMPIEZA DE NÚMEROS
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce').fillna(0)
df['cantidad_eliminada'] = pd.to_numeric(df['cantidad_eliminada'], errors='coerce').fillna(0)

# 6. LÓGICA DE TRABAJO: Ordenar cronológicamente
# Para tu lógica de "dura hasta la próxima apertura", el orden es sagrado
df = df.sort_values(by=['antibiotico', 'fecha_apertura'], ascending=[True, True]).reset_index(drop=True)

# 7. VERIFICACIÓN DE RESULTADOS
# print("📊 Resumen del Inventario Limpio:")
# print(df.info()) # Aquí verás cuántos NaT quedaron en cada columna
# print("\n--- Vista de los datos procesados ---")
# print(df.fillna("---"))

# 1. Calculamos el total RECEPCIONADO por antibiótico
# Filtramos solo las filas de tipo 'Recepción' y sumamos su columna 'cantidad'
recepcionados = df[df['tipo_registro'] == 'Recepción'].groupby('antibiotico')['cantidad'].sum()

# 2. Calculamos las APERTURAS
# Cada registro de tipo 'Apertura' resta 1 unidad al stock (o un tubo)
aperturas = df[df['tipo_registro'] == 'Apertura'].groupby('antibiotico').size()

# 3. Calculamos lo ELIMINADO
# Sumamos la columna 'cantidad_eliminada' por cada antibiótico
eliminados = df.groupby('antibiotico')['cantidad_eliminada'].sum()

# 4. UNIMOS TODO EN UNA SOLA TABLA DE RESUMEN
# Usamos .fillna(0) porque si un antibiótico no tiene aperturas o eliminados, queremos que sea 0, no NaN
resumen_stock = pd.DataFrame({
    'Recepcionados': recepcionados,
    'Aperturas': aperturas,
    'Eliminados': eliminados
}).fillna(0)

# 5. APLICAMOS TU LÓGICA DE STOCK
resumen_stock['Stock_Actual'] = resumen_stock['Recepcionados'] - resumen_stock['Aperturas'] - resumen_stock['Eliminados']

# 6. ORDENAR POR STOCK BAJO (Prioridad para reposición)
resumen_stock = resumen_stock.sort_values(by='Stock_Actual', ascending=True)

print("📊 RESUMEN DE STOCK ACTUAL - LABORATORIO")
print(resumen_stock)

📊 RESUMEN DE STOCK ACTUAL - LABORATORIO
                                          Recepcionados  Aperturas  \
antibiotico                                                          
Amikacina (E-test)                                  1.0        1.0   
Azitromicina (E-test)                               1.0        1.0   
Aztreonam (E-test)                                  2.0        1.0   
Ceftazidima (E-test)                                1.0        0.0   
Imipenem (E-test)                                   2.0        1.0   
Ciprofloxacino (E-test)                             2.0        1.0   
Ceftolozano/Tazobactam (E-test)                     2.0        1.0   
Ceftazidima/Avibactam (E-test)                      2.0        1.0   
Vancomicina (E-test)                                1.0        0.0   
Meropenem (E-test)                                  1.0        0.0   
Linezolid (Sensidisco)                              3.0        2.0   
Levofloxacino (E-test)                            